# ZVault · Quantum-safe AI Data Vault demo (IBM LinuxONE)

Run from the `vault/` folder, top to bottom (**Run → Run All Cells**). Needs **OpenSSL 3.5+** (ML-KEM, ML-DSA).
If the system OpenSSL is older, point `ZVAULT_LIBCRYPTO` at a 3.5 `libcrypto.so.3`.

| Part | What happens |
|---|---|
| **1. Test** | Runs the automatic tests |
| **2. Setup** | Checks the machine (CPACF, OpenSSL) and picks the embedding model |
| **3. Quantum-safe keys** | RSA vs hybrid ML-KEM-768 + X25519, per-customer keys, legal hold, speed |
| **4. Data** | Synthetic bank stored as encrypted AI copies |
| **5. Forget on request** | Recovery test, ML-DSA certificate, transparency log |
| **6. Forget on schedule** | Retention model + scheduler |
| **7. Summary** | What is real vs simulated |

All customers and numbers are synthetic (generic names only).

# 1. Run the tests

In [ ]:
%pip install -q pytest numpy scikit-learn pandas

In [ ]:
import sys
!{sys.executable} -m pytest -q -p no:cacheprovider

# 2. Setup

`USE_GTR = True` uses the real `gtr-t5-base` embedding model if `sentence-transformers` is installed
(`%pip install sentence-transformers`). Otherwise a numpy embedder is used and the notebook says so.
`TRY_VEC2TEXT = True` tries the generative vec2text inversion attack (large download; may not install on s390x).

In [ ]:
import sys, time, shutil, platform, copy
from pathlib import Path
PROJECT = Path.cwd()  # run this notebook from the vault/ folder
sys.path.insert(0, str(PROJECT))
try:
    display
except NameError:
    display = print

import numpy as np, pandas as pd
from zvault_vault.crypto import ossl, hybrid
from zvault_vault.data import generate_bank, RETENTION_DAYS
from zvault_vault.embeddings import default_embedder
from zvault_vault.keys import SoftHSM, KeyState, KeyDestroyedError, KeyNotActiveError, LegalHoldError
from zvault_vault.store import EncryptedStore
from zvault_vault.ingest import ingest_bank
from zvault_vault.broker import AccessBroker, AccessDenied
from zvault_vault.assistant import AnalystAssistant
from zvault_vault.certs import CertificateSigner, verify_certificate
from zvault_vault.tlog import TransparencyLog, verify_inclusion, verify_tree_head
from zvault_vault.attacks import CandidateInverter, Vec2TextInverter, auc, membership_scores, inversion_result, specific_texts
from zvault_vault.erasure import ErasureEngine
from zvault_vault.retention import RetentionModel, RetentionScheduler, DAY

USE_GTR = True
TRY_VEC2TEXT = False

cpacf = "msa" in Path("/proc/cpuinfo").read_text() if Path("/proc/cpuinfo").exists() else False
print("Machine     :", platform.machine())
print("OpenSSL     :", ossl.version())
print("CPACF (AES) :", "present" if cpacf else "not detected (not an IBM Z CPU)")
embedder = default_embedder(prefer_gtr=USE_GTR)
print("Embedder    :", embedder.name)

# 3. Quantum-safe keys

### 1. Quantum-vulnerable vs quantum-safe key wrapping

The same 32-byte customer key, wrapped two ways. An attacker who records both blobs today and gets a
large quantum computer later can break RSA with Shor's algorithm. Breaking the hybrid needs **both**
X25519 and ML-KEM-768 broken, and ML-KEM is designed to resist quantum attacks.

In [ ]:
customer_key = ossl.random_bytes(32)
context = b"kek|CUST-0001|v1"

rsa = ossl.PKey.generate("RSA", rsa_bits=2048)
rsa_blob = ossl.rsa_oaep_encrypt(rsa, customer_key)

root = hybrid.HybridPrivateKey.generate()
pq_blob = hybrid.wrap_key(root.public_key(), customer_key, context)

assert ossl.rsa_oaep_decrypt(rsa, rsa_blob) == customer_key
assert hybrid.unwrap_key(root, pq_blob, context) == customer_key

print(f"{'Wrapping':<34}{'Size':>8}   Recorded today, readable by a future quantum computer?")
print(f"{'RSA-2048 OAEP':<34}{len(rsa_blob):>6} B   YES  (Shor's algorithm breaks RSA)")
print(f"{'Hybrid X25519 + ML-KEM-768':<34}{len(pq_blob):>6} B   NO   (needs ML-KEM-768 broken too)")

### 2. Customer keys and encrypted AI copies

A software stand-in plays the Crypto Express HSM here (this LinuxONE has no adapter); the interface is the same.
Each AI copy is encrypted the moment it is stored and registered in the Copy Registry.

In [ ]:
DEMO = PROJECT / "demo_data" / "nb01"
shutil.rmtree(DEMO, ignore_errors=True)

hsm = SoftHSM(DEMO / "hsm")
store = EncryptedStore(DEMO / "copies.db", hsm)
print("Key backend:", hsm.backend_label)

for customer in ("CUST-0001", "CUST-0002"):
    hsm.create_customer_key(customer)

with hsm.lease("CUST-0001", purpose="card dispute D-4471") as lease:
    copies_1 = [
        store.put_text(lease, "chat", "I want to apply for a home loan of 50 lakh."),
        store.put_text(lease, "embedding", "[768-dim vector of the home loan chat]"),
        store.put_text(lease, "agent-memory", "Customer disputes card transaction of Rs 18,400 on 12 Oct."),
    ]
    print("CUST-0001 reads back:", store.get_text(lease, copies_1[0]))

with hsm.lease("CUST-0002", purpose="card dispute D-4502") as lease:
    copy_2 = store.put_text(lease, "chat", "Please block my debit card.")

for r in store.registry.for_customer("CUST-0001"):
    print(f"  registered {r.copy_id:<26} kind={r.kind:<13} location={r.location}")

In [ ]:
sealed = store.sealed(copies_1[0])
print("What someone who copies the disk gets:")
print("  ciphertext :", sealed["ciphertext"][:32].hex(), "...")
print("  wrapped key:", sealed["wrapped_dek"][:32].hex(), "...")
print("  plaintext visible on disk?", b"home loan" in (DEMO / "copies.db").read_bytes())

### 3. Forget: destroy one customer's key

The key goes `ACTIVE → PENDING_DESTRUCTION → DESTROYED` (AWS KMS-style). Once destroyed, every AI copy of
that customer is unreadable at once, without finding each copy. Other customers are unaffected.

In [ ]:
print("Before:", hsm.info("CUST-0001").state.value)
hsm.schedule_destruction("CUST-0001", delay_seconds=0)
print("Pending:", hsm.info("CUST-0001").state.value)
print("Destroyed now:", hsm.destroy_due())
print("After:", hsm.info("CUST-0001").state.value, "| wrapped key still stored:", hsm.wrapped_kek("CUST-0001") is not None)

try:
    hsm.lease("CUST-0001", purpose="card dispute D-4471")
except KeyDestroyedError as e:
    print("Reading CUST-0001's copies ->", type(e).__name__, ":", e)

print("Ciphertext of CUST-0001 still on disk:", all(store.sealed(c)["ciphertext"] for c in copies_1), "(but no key exists to open it)")

with hsm.lease("CUST-0002", purpose="card dispute D-4502") as lease:
    print("CUST-0002 unaffected:", store.get_text(lease, copy_2))

### 4. Legal hold wins

A customer under an open fraud investigation cannot erase their data (PMLA retention). The hold blocks
destruction, even if it is placed during the pending window.

In [ ]:
hsm.set_legal_hold("CUST-0002", True)
try:
    hsm.schedule_destruction("CUST-0002")
except LegalHoldError as e:
    print("Erasure request ->", type(e).__name__, ":", e)
print("CUST-0002 state:", hsm.info("CUST-0002").state.value)

### 5. Speed on this machine

Measured, not estimated. On IBM Z, OpenSSL runs AES-GCM on the CPACF instructions.

In [ ]:
def bench(fn, seconds=1.0):
    n, start = 0, time.perf_counter()
    while time.perf_counter() - start < seconds:
        fn(); n += 1
    return n / (time.perf_counter() - start)

key, mb = ossl.random_bytes(32), ossl.random_bytes(1 << 20)
blob = ossl.aes_gcm_encrypt(key, mb)
mlkem, mldsa = ossl.PKey.generate("ML-KEM-768"), ossl.PKey.generate("ML-DSA-65")
ct, _ = ossl.kem_encapsulate(mlkem)
sig = ossl.sign(mldsa, b"certificate")
pub = root.public_key()
wrapped = hybrid.wrap_key(pub, key, b"ctx")

results = [
    ("AES-256-GCM encrypt 1 MiB", bench(lambda: ossl.aes_gcm_encrypt(key, mb)), "MiB/s"),
    ("AES-256-GCM decrypt 1 MiB", bench(lambda: ossl.aes_gcm_decrypt(key, blob)), "MiB/s"),
    ("ML-KEM-768 encapsulate", bench(lambda: ossl.kem_encapsulate(mlkem)), "ops/s"),
    ("ML-KEM-768 decapsulate", bench(lambda: ossl.kem_decapsulate(mlkem, ct)), "ops/s"),
    ("Hybrid wrap customer key", bench(lambda: hybrid.wrap_key(pub, key, b"ctx")), "ops/s"),
    ("Hybrid unwrap customer key", bench(lambda: hybrid.unwrap_key(root, wrapped, b"ctx")), "ops/s"),
    ("ML-DSA-65 sign", bench(lambda: ossl.sign(mldsa, b"certificate")), "ops/s"),
    ("ML-DSA-65 verify", bench(lambda: ossl.verify(mldsa, b"certificate", sig)), "ops/s"),
    ("RSA-2048 OAEP decrypt (classical)", bench(lambda: ossl.rsa_oaep_decrypt(rsa, rsa_blob)), "ops/s"),
]
print(f"{'Operation':<36}{'Rate':>12}")
for name, rate, unit in results:
    print(f"{name:<36}{rate:>10,.0f} {unit}")

In [ ]:
store.close(); hsm.close()

# 4. Synthetic bank data → encrypted AI copies

In [ ]:
DEMO = PROJECT / "demo_data" / "main"
shutil.rmtree(DEMO, ignore_errors=True)
bank = generate_bank(60, seed=7)
df = pd.DataFrame([m.__dict__ for m in bank.messages])
print(len(bank.customers), "customers,", len(df), "messages")
display(df.groupby(["kind", "retention"]).size().unstack(fill_value=0))
display(df.sample(6, random_state=1)[["customer_id", "kind", "retention", "text"]])

In [ ]:
NOW = time.time()
hsm = SoftHSM(DEMO / "hsm")
store = EncryptedStore(DEMO / "copies.db", hsm)
t0 = time.perf_counter()
ingest_bank(bank, hsm, store, embedder, now=NOW, verbose=True)
print(f"Stored {len(store.registry.all())} encrypted AI copies (text + embedding) in {time.perf_counter() - t0:.1f} s")
disk = (DEMO / "copies.db").read_bytes()
print("Any customer name or PAN readable on disk?", any(c.name.encode() in disk or c.pan.encode() in disk for c in bank.customers))

# 5. Forget on request

### 5.1 The problem: a conventional AI pipeline
A normal vector database keeps plaintext embeddings and "deletes" by flagging them (Ghost Vectors, arXiv 2606.18497).
An attacker who reads the vectors can turn them back into text.

In [ ]:
outsiders = generate_bank(80, seed=99, prefix="AUX")
inverter = CandidateInverter(embedder, [m.text for m in outsiders.messages])
if TRY_VEC2TEXT:
    try:
        inverter = Vec2TextInverter()
    except Exception as e:
        print("vec2text unavailable:", type(e).__name__, "- using dictionary inversion")
print("Inversion attack:", inverter.name)

target = bank.customers[0]
msgs = bank.for_customer(target.customer_id)
plain_vectors = embedder.encode([m.text for m in msgs])   # conventional pipeline: stored in plaintext
soft_deleted = True                                        # "deleted" = a flag; vectors stay on disk
leak = inversion_result(inverter, plain_vectors, [m.text for m in msgs], target.identifiers())
print(f"Conventional pipeline after 'delete': attacker recovers text with token F1 = {leak['mean_token_f1']}")
for m, g in list(zip(msgs, inverter.invert(plain_vectors)))[:3]:
    print("  original :", m.text)
    print("  recovered:", g)

### 5.2 The analyst assistant, through the broker

In [ ]:
broker = AccessBroker(hsm, DEMO / "broker.db")
broker.register_analyst("A-2291", "Fraud analyst", mfa_verified=True)
case = [bank.customers[0].customer_id, bank.customers[1].customer_id]
broker.open_case("D-4471", "card dispute", case)
assistant = AnalystAssistant(broker, store, embedder)

def show(ans):
    print("Q:", ans.question)
    for score, cust, kind, text in ans.sources:
        print(f"  {score:.2f}  {cust}  [{kind}]  {text}")
    for cust, why in ans.blocked:
        print(f"  BLOCKED {cust}: {why}")

show(assistant.ask("A-2291", "D-4471", "customer disputes a payment they did not make"))
try:
    broker.request("A-2291", "D-4471", bank.customers[5].customer_id)
except AccessDenied as e:
    print("Customer outside the case ->", e)

### 5.3 The customer taps **Forget me**

In [ ]:
signer = CertificateSigner(DEMO / "enclave")
log = TransparencyLog(DEMO / "tlog.db", signer)
probes = [m.text for m in generate_bank(40, seed=123, prefix="NON").messages]  # people who were never customers
engine = ErasureEngine(hsm, store, signer, log, embedder, inverter, [DEMO], probes)

t0 = time.perf_counter()
out = engine.erase(target.customer_id, identity_verified=True, identifiers=target.identifiers())
cert = out["certificate"]
rt = cert["recovery_test"]
print(f"Erasure {cert['request_id']} for {cert['customer_id']}: {cert['status'].upper()}  ({time.perf_counter() - t0:.1f} s)")
print("Key:", cert["key"]["state"], "|", cert["key"]["backend"])
print(pd.Series([c["result"] for c in cert["copies"]]).value_counts().to_string())
inv, mem = rt["inversion"], rt["membership_inference"]
print(pd.DataFrame({
    "before erasure": [inv["before_erasure"]["mean_token_f1"], "-",
                       mem["before_erasure"]["share_flagged"], mem["before_erasure"]["detected"]],
    "after erasure":  [inv["after_erasure"]["mean_token_f1"],
                       "{mean_token_f1} (95% up to {p95_token_f1})".format(**inv["after_erasure"]["random_noise_baseline"]),
                       mem["after_erasure"]["share_flagged"], mem["after_erasure"]["detected"]],
}, index=["inversion: token F1", "same attack on random noise", "membership: share of messages flagged",
          "membership: customer detected"]).to_string())
print("Identifiers recovered after erasure:", inv["after_erasure"]["identifiers_recovered"] or "none")
print("Residue scan:", rt["residue_scan"]["files_scanned"], "files,", len(rt["residue_scan"]["hits"]), "readable leaks")
print("Not verifiable:", [c["location"] for c in cert["copies"] if c["result"] == "not_verifiable"])

### 5.4 The analyst asks again

In [ ]:
show(assistant.ask("A-2291", "D-4471", "customer disputes a payment they did not make"))

### 5.5 Anyone can verify the certificate, without trusting the bank

In [ ]:
head = out["tree_head"]
print("Certificate signature (ML-DSA-65) valid:", verify_certificate(cert, signer.public_der))
print("Signed tree head valid               :", verify_tree_head(head, signer.public_der))
print("Certificate is in the public log     :", verify_inclusion(log.entry(out["log_index"]), out["log_index"], head.size, out["inclusion_proof"], head.root))
forged = copy.deepcopy(cert); forged["status"] = "erased"; forged["copies"] = forged["copies"][:1]
print("Edited certificate still valid?      :", verify_certificate(forged, signer.public_der))
print("Enclave measurement:", cert["enclave"]["measurement_sha3_256"][:24], "...", "|", cert["enclave"]["attestation"])

### 5.6 Membership inference over a batch of erased customers
AUC 1.0 = attacker always tells members apart; 0.5 = guessing.

In [ ]:
batch = bank.customers[10:15]
members = specific_texts([m.text for c in batch for m in bank.for_customer(c.customer_id)])
non_members = specific_texts(probes)
index = engine.readable_index()
before = auc(membership_scores(embedder, index, members), membership_scores(embedder, index, non_members))
for c in batch:
    engine.erase(c.customer_id, True, c.identifiers())
index = engine.readable_index()
after = auc(membership_scores(embedder, index, members), membership_scores(embedder, index, non_members))
print(f"Membership-inference AUC  before erasure: {before:.2f}   after erasure: {after:.2f}")

### 5.7 Legal hold: erasure refused, and the customer is told why

In [ ]:
held = bank.customers[2]
hsm.set_legal_hold(held.customer_id, True)
refusal = engine.erase(held.customer_id, True, held.identifiers())["certificate"]
print(refusal["status"], "-", refusal["reason"])
print("Signed:", verify_certificate(refusal, signer.public_der), "| key state:", hsm.info(held.customer_id).state.value)

# 6. Forget on schedule (QShelf)

The model predicts how long each AI copy must be kept: **short** 30 days · **service** 1 year · **regulatory** 5 years · **contract** 20 years.
It is evaluated on **phrasings it never saw in training**.

Prior art: Wells Fargo patent US 11,240,014 B1 assigns shelf-life from regulatory retention tables.
QShelf predicts it from the content of AI-derived copies, which no retention table lists.

In [ ]:
from sklearn.metrics import classification_report
train = [m for m in generate_bank(300, seed=11).messages if m.family == "train"]
test = [m for m in generate_bank(300, seed=12).messages if m.family == "test"]
t0 = time.perf_counter()
model = RetentionModel().fit([m.text for m in train], [m.kind for m in train], [m.retention for m in train])
print(f"Trained on {len(train)} copies in {time.perf_counter() - t0:.1f} s; testing on {len(test)} unseen phrasings")
preds = model.predict([m.text for m in test], [m.kind for m in test])
y, p = [m.retention for m in test], [x.retention_class for x in preds]
print(classification_report(y, p, digits=3))
display(pd.crosstab(pd.Series(y, name="true"), pd.Series(p, name="predicted")))
print("Flagged for human review (low confidence, longer class kept):", sum(x.needs_review for x in preds))
too_early = sum(RETENTION_DAYS[q] < RETENTION_DAYS[t] for q, t in zip(p, y))
print("Copies that would be deleted earlier than required:", too_early, "of", len(y))
model.save(DEMO / "qshelf.pkl")

In [ ]:
for text, kind in [("Please block my debit card, I lost it", "chat"),
                   ("I would like to borrow 40 lakh to buy a flat, tenure 20 years.", "chat"),
                   ("mera balance kitna hai?", "chat")]:
    pr = model.predict([text], [kind])[0]
    words = ", ".join(w for w, _ in model.explain(text, kind, top=4))
    print(f"{pr.retention_class:<10} {pr.days:>5} days  conf {pr.confidence:.2f}  because: {words:<45} | {text}")

### 6.1 The scheduler on the live store, with simulated time

In [ ]:
clock = {"now": NOW}
scheduler = RetentionScheduler(store, hsm, model, signer, log, clock=lambda: clock["now"])
assigned = scheduler.assign()
print("Retention assigned to", len(assigned), "text copies (their embeddings get the same expiry)")
display(pd.Series([a["retention_class"] for a in assigned]).value_counts())

for days in (45, 400, 2000):
    clock["now"] = NOW + days * DAY
    rep = scheduler.run()
    by_class = pd.Series([s["retention_class"] for s in rep["shredded"]]).value_counts().to_dict()
    print(f"+{days:>4} days: shredded {len(rep['shredded']):>3} copies {by_class}  | kept for legal hold: {len(rep['kept_for_legal_hold'])}")
    if "record" in rep:
        assert verify_certificate(rep["record"], signer.public_der)

In [ ]:
clock["now"] = NOW
broker.open_case("D-5001", "account review", [bank.customers[3].customer_id])
show(assistant.ask("A-2291", "D-5001", "what did the customer ask about"))
print("(expired copies are shredded and no longer appear)")

# 7. Summary

In [ ]:
head = log.tree_head()
ok = all(verify_inclusion(log.entry(i), i, head.size, log.inclusion_proof(i), head.root) for i in range(head.size))
print(f"Transparency log: {head.size} signed records | tree head valid: {verify_tree_head(head, signer.public_der)} | every inclusion proof valid: {ok}")
print()
print(pd.DataFrame([
    ["AES-256-GCM on CPACF", "real" if cpacf else "real (x86 here)", "OpenSSL 3.5 system library"],
    ["ML-KEM-768 + X25519 hybrid key wrapping", "real", "OpenSSL 3.5"],
    ["ML-DSA-65 certificates and log", "real", "OpenSSL 3.5"],
    ["Crypto Express HSM", "simulated", hsm.backend_label],
    ["IBM Secure Execution attestation", "simulated", "measurement computed in software"],
    ["Embedding model", "real" if "gtr" in embedder.name else "fallback", embedder.name],
    ["Inversion attack", "real", inverter.name],
], columns=["Part", "Status", "Detail"]).to_string(index=False))
store.close(); hsm.close(); broker.close(); log.close()